In [73]:
import sys

import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score,classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [74]:
def carrega_dados(file_name, separador=',', drop_cols=None):
    data = pd.read_csv(f"../data/{file_name}", sep=separador)
    print("Reorganizando os dados...")
    data = data.sample(frac=1)

    try:
        data = data if not drop_cols else data.drop(drop_cols, axis=1)
    except KeyError:
        print(f"Colunas {drop_cols} não encontradas no DataFrame.")
        sys.exit(1)

    print("Dados carregados")
    return data

def encoding_cols(data, cols):
    print("Transformando colunas categóricas em numéricas...")

    if not data.columns.isin(cols).any():
        print(f"Colunas {cols} não encontradas no DataFrame.")
        raise KeyError(f"Colunas {cols} não encontradas no DataFrame.")
    

    label_encoder = LabelEncoder()

    for col in cols:
        if data[col].dtype == 'object':
            data[col] = label_encoder.fit_transform(data[col])
        else:
            print(f"A coluna {col} não é categórica, não será transformada.")
            raise ValueError(f"A coluna {col} não é categórica, não será transformada.")
        
    print("Colunas categóricas transformadas em numéricas")
    
    return data

def standardize_data(X_train, X_test):
    print("Normalizandos os dados com z-score")
    scaler = StandardScaler()

    if not isinstance(X_train, pd.DataFrame) or not isinstance(X_test, pd.DataFrame):
        raise ValueError("X_train e X_test devem ser DataFrames do pandas.")
    
    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError("X_train e X_test devem ter o mesmo número de colunas.")
    
    if X_train.isnull().values.any() or X_test.isnull().values.any():
        raise ValueError("X_train e X_test não podem conter valores nulos.")

    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)
    print("Dados normalizados com z-score")

    scaler_filename = 'scaler.pkl'
    with open(f"../artifacts/{scaler_filename}", 'wb') as file:
        pickle.dump(scaler, file)
    return X_train, X_test

def standardize_data_from_file(data, scaler_filename):
    print("Normalizando os dados com z-score")
    scaler = pickle.load(open(f"../artifacts/{scaler_filename}", 'rb'))

    if not isinstance(data, pd.DataFrame):
        raise ValueError("data devem ser DataFrames do pandas.")
    
    if data.isnull().values.any():
        raise ValueError("data não pode conter valores nulos.")

    data = scaler.transform(data)
    print("Dados normalizados com z-score")

    return data

def save_model(model, filename):
    with open(filename, 'wb') as file:
        pickle.dump(model, file)
    print("Modelo salvo")

def load_model(filename):
    with open(filename, 'rb') as file:
        model = pickle.load(file)
    return model

def get_x_y(data, y_label, x_cols=None):
    print("Preparando amostras de treino e validação")
    try:
        X = data.drop(y_label, axis=1)
        X = X if not x_cols else X[x_cols]
        y = data[y_label]
    except KeyError:
        print(f"Colunas {y_label} ou {x_cols} não encontradas no DataFrame.")
        sys.exit(1)

    print("Amostras de treino e validação preparadas")

    return X, y


#### Carregamento

In [75]:
data = carrega_dados("heart_disease_dataset.csv", drop_cols=["Gender"])
data_train, data_test = train_test_split(data, test_size=0.1)

Reorganizando os dados...
Dados carregados


#### Split de treinamento

In [76]:
X, y = get_x_y(data_train, "Heart Disease", x_cols=[])

Preparando amostras de treino e validação
Amostras de treino e validação preparadas


In [77]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

#### Pré-processamento

In [78]:
X_train = encoding_cols(X_train, X_train.select_dtypes(include=['object']).columns.tolist())
X_val = encoding_cols(X_val, X_val.select_dtypes(include=['object']).columns.tolist())

X_train, X_val = standardize_data(X_train, X_val)


Transformando colunas categóricas em numéricas...
Colunas categóricas transformadas em numéricas
Transformando colunas categóricas em numéricas...
Colunas categóricas transformadas em numéricas
Normalizandos os dados com z-score
Dados normalizados com z-score


#### Treinamento

In [79]:
arvore = tree.DecisionTreeClassifier()
arvore.fit(X_train, y_train)
arvore

DecisionTreeClassifier()

In [80]:
knn = KNeighborsClassifier(metric="euclidean", n_neighbors=3)
knn.fit(X_train, y_train)
knn

KNeighborsClassifier(metric='euclidean', n_neighbors=3)

In [81]:
log_reg = LogisticRegression(solver="sag")
log_reg.fit(X_train, y_train)
log_reg

LogisticRegression(solver='sag')

#### Validação

In [82]:
y_pred_arvore = arvore.predict(X_val)
y_pred_knn = knn.predict(X_val)
y_pred_log_reg = log_reg.predict(X_val)

In [83]:
confusion_matrix_arvore = confusion_matrix(y_val, y_pred_arvore)
classification_report_arvore = classification_report(y_val, y_pred_arvore)


In [84]:
print(confusion_matrix_arvore)
print(classification_report_arvore)

[[102   0]
 [  0  78]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       102
           1       1.00      1.00      1.00        78

    accuracy                           1.00       180
   macro avg       1.00      1.00      1.00       180
weighted avg       1.00      1.00      1.00       180



In [85]:
confusion_matrix_knn = confusion_matrix(y_val, y_pred_knn)
classification_report_knn = classification_report(y_val, y_pred_knn)

In [86]:
print(confusion_matrix_knn)
print(classification_report_knn)

[[88 14]
 [24 54]]
              precision    recall  f1-score   support

           0       0.79      0.86      0.82       102
           1       0.79      0.69      0.74        78

    accuracy                           0.79       180
   macro avg       0.79      0.78      0.78       180
weighted avg       0.79      0.79      0.79       180



In [87]:
confusion_matrix_log_reg = confusion_matrix(y_val, y_pred_log_reg)
classification_report_log_reg = classification_report(y_val, y_pred_log_reg)

In [88]:
print(confusion_matrix_log_reg)
print(classification_report_log_reg)

[[87 15]
 [15 63]]
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       102
           1       0.81      0.81      0.81        78

    accuracy                           0.83       180
   macro avg       0.83      0.83      0.83       180
weighted avg       0.83      0.83      0.83       180



#### Comparação e análise de erros

In [89]:
data_test_transformed = encoding_cols(data_test, data_test.select_dtypes(include=['object']).columns.tolist())
data_test_transformed = data_test_transformed.drop("Heart Disease", axis=1)
data_test_transformed = standardize_data_from_file(data_test_transformed, "scaler.pkl")
y_test = data_test["Heart Disease"]

Transformando colunas categóricas em numéricas...
Colunas categóricas transformadas em numéricas
Normalizando os dados com z-score
Dados normalizados com z-score


In [90]:
y_arvore_teste = arvore.predict(data_test_transformed)
y_knn_teste = knn.predict(data_test_transformed)
y_log_reg_teste = log_reg.predict(data_test_transformed)


In [92]:
data_test["Respostas Arvore"] = y_arvore_teste
data_test["Respostas KNN"] = y_knn_teste
data_test["Respostas Log Reg"] = y_log_reg_teste

In [95]:
data_test.head(20)

,Age,Cholesterol,Blood Pressure,Heart Rate,Smoking,Alcohol Intake,Exercise Hours,Family History,Diabetes,Obesity,Stress Level,Blood Sugar,Exercise Induced Angina,Chest Pain Type,Heart Disease,Respostas Arvore,Respostas KNN,Respostas Log Reg
843,31,333,168,93,0,1,4,0,0,0,6,184,0,0,0,0,0,0
603,56,274,105,79,0,1,3,0,0,1,5,174,0,2,1,1,1,1
940,64,333,113,63,0,2,2,1,1,0,3,95,1,0,1,1,1,1
379,46,328,99,85,2,1,7,0,0,1,7,115,1,3,0,0,0,0
909,56,247,113,76,2,0,1,1,0,0,9,86,1,0,1,1,1,0
594,52,156,117,63,2,1,5,0,0,1,7,116,0,2,0,0,0,0
335,70,183,158,76,2,1,7,0,1,0,1,135,1,0,0,0,1,0
482,44,296,99,63,2,0,6,1,1,0,7,185,1,0,0,0,0,0
395,31,177,145,65,0,2,5,0,0,0,5,110,1,2,0,0,0,0
951,58,312,134,79,0,0,3,1,0,0,7,198,1,1,1,1,1,1


In [96]:
data_test.to_csv("../data/comparativos_modelos.csv", index=False)